In [ ]:
"""
Cash Flow Prediction Script
Loads pre-trained models and makes predictions on new data
"""

import numpy as np
import pandas as pd
import joblib
import pickle
from tensorflow.keras.models import load_model
import rqdatac as rq

# Initialize rqdatac
rq.init()

def load_all_models(model_dir='.'):
    """
    Load all trained models and preprocessing objects
    
    Parameters:
    - model_dir: directory where models are stored
    
    Returns:
    - Dictionary containing all loaded models and utilities
    """
    print("Loading models...")
    
    models = {
        # Random Forest
        'ffcf_rf_model': joblib.load(f'{model_dir}/ffcf_random_forest_model.pkl'),
        'ffcf_rf_selector': joblib.load(f'{model_dir}/ffcf_rf_selector.pkl'),
        
        # Gradient Boosting
        'ffcf_gbr_model': joblib.load(f'{model_dir}/ffcf_gradient_boosting_model.pkl'),
        
        # LSTM
        'ffcf_lstm_model': load_model(f'{model_dir}/ffcf_lstm_model.h5'),
        
        # Scalers
        'ffcf_x_scaler': joblib.load(f'{model_dir}/ffcf_x_scaler.pkl'),
        'ffcf_y_scaler': joblib.load(f'{model_dir}/ffcf_y_scaler.pkl'),
    }
    
    # Load configuration
    with open(f'{model_dir}/ffcf_model_config.pkl', 'rb') as f:
        config = pickle.load(f)
    models['config'] = config
    
    print("✓ All models loaded successfully!")
    return models

def prepare_features(financial_df):
    """
    Prepare financial data for prediction
    
    Parameters:
    - financial_df: DataFrame with financial data for a single stock
    
    Returns:
    - numpy array ready for model input
    """
    # Group by year and take last value
    X = financial_df.groupby([
        pd.Grouper(level='order_book_id'),
        pd.Grouper(level='date', freq='Y')
    ]).last().fillna(0)
    
    X = X.reorder_levels(['order_book_id', 'date'])
    
    # Keep only latest 5 years
    X = X.tail(5)
    
    # Flatten for ML models
    prediction_input = X.values.flatten().reshape(1, -1)
    
    return X, prediction_input

def predict_random_forest(models, features):
    """Make prediction using Random Forest model"""
    # Apply feature selector
    features_selected = models['ffcf_rf_selector'].transform(features)
    prediction = models['ffcf_rf_model'].predict(features_selected)
    return prediction

def predict_gradient_boosting(models, features):
    """Make prediction using Gradient Boosting model"""
    prediction = models['ffcf_gbr_model'].predict(features)
    return prediction

def predict_lstm(models, features):
    """Make prediction using LSTM model"""
    # Scale features
    features_scaled = models['ffcf_x_scaler'].transform(features)
    
    # Reshape for LSTM (samples, timesteps, features)
    n_features = features_scaled.shape[1] // 5
    features_lstm = features_scaled.reshape(
        features_scaled.shape[0],
        5,
        n_features
    )
    
    # Predict and inverse scale
    prediction_scaled = models['ffcf_lstm_model'].predict(features_lstm, verbose=0)
    prediction = models['ffcf_y_scaler'].inverse_transform(prediction_scaled)
    
    return prediction

def predict_all_models(models, features):
    """
    Make predictions using all models
    
    Returns:
    - Dictionary with predictions from each model
    """
    predictions = {}
    
    # Random Forest
    predictions['ffcf_random_forest'] = predict_random_forest(models, features)
    
    # Gradient Boosting
    predictions['ffcf_gradient_boosting'] = predict_gradient_boosting(models, features)
    
    # LSTM
    predictions['ffcf_lstm'] = predict_lstm(models, features)
    
    return predictions

def predict_for_stock(models, stock_code, factors, start_date='2022-01-01', end_date='2026-05-01'):
    """
    Complete prediction pipeline for a specific stock
    
    Parameters:
    - models: loaded models dictionary
    - stock_code: stock symbol (e.g., '601318.XSHG')
    - factors: list of factor names
    - start_date: start date for fetching data
    - end_date: end date for fetching data
    
    Returns:
    - Dictionary with predictions and metadata
    """
    print(f"\nFetching financial data for {stock_code}...")
    
    # Fetch financial data
    financial_data = rq.get_factor(
        stock_code, 
        factors, 
        start_date=start_date, 
        end_date=end_date
    )
    
    financial_data = financial_data.fillna(0)
    
    # Prepare features
    X_df, features = prepare_features(financial_data)
    
    # Make predictions
    predictions = predict_all_models(models, features)
    
    # Format results
    years = [f'Year+{i+1}' for i in range(5)]
    
    results = {
        'stock_code': stock_code,
        'features_used': X_df,
        'predictions': predictions,
        'years': years
    }
    
    return results

def display_predictions(results):
    """Display predictions in a readable format"""
    print(f"\n{'='*60}")
    print(f"Cash Flow Predictions for {results['stock_code']}")
    print(f"{'='*60}")
    
    for model_name, preds in results['predictions'].items():
        print(f"\n{model_name.upper()}:")
        print("-" * 40)
        for year, value in zip(results['years'], preds[0]):
            print(f"  {year}: {value:,.2f}")

def calculate_dcf(predictions, discount_rate=0.20, growth_rate=0.15):
    """
    Calculate DCF valuation from predictions
    
    Parameters:
    - predictions: array of future cash flows [CF1, CF2, CF3, CF4, CF5]
    - discount_rate: required return (e.g., 0.20 for 20%)
    - growth_rate: terminal growth rate (e.g., 0.15 for 15%)
    """
    def terminal_value_gordon(cf_final_year, discount_rate, growth_rate):
        return cf_final_year * (1 + growth_rate) / (discount_rate - growth_rate)
    
    def dcf_valuation(cash_flows, discount_rate, terminal_value=None):
        cash_flows = np.array(cash_flows, dtype=float)
        years = np.arange(1, len(cash_flows) + 1)
        discounted_cf = cash_flows / (1 + discount_rate) ** years
        pv = discounted_cf.sum()
        
        if terminal_value is not None:
            pv_terminal = terminal_value / (1 + discount_rate) ** len(cash_flows)
            pv += pv_terminal
        
        return pv
    
    terminal = terminal_value_gordon(predictions[-1], discount_rate, growth_rate)
    dcf_value = dcf_valuation(predictions, discount_rate, terminal)
    
    return dcf_value

def display_dcf_results(results, discount_rate=0.20, growth_rate=0.15):
    """Display DCF valuations for all models"""
    print(f"\n{'='*60}")
    print(f"DCF VALUATION (Discount Rate: {discount_rate*100}%, Growth Rate: {growth_rate*100}%)")
    print(f"{'='*60}")
    
    for model_name, preds in results['predictions'].items():
        dcf_value = calculate_dcf(preds[0], discount_rate, growth_rate)
        print(f"\n{model_name.upper()}:")
        print(f"  DCF Value: {dcf_value:,.2f}")

# =========================
# Example Usage
# =========================

if __name__ == "__main__":
    # Define factors (same as in training)
    factors = [
        # revenue
        'revenue',
        'operating_revenue',
        'net_operating_revenue',
        
        # valuation
        'book_value_per_share',
        'book_value_per_share_lf',
        'book_value_per_share_ttm',

        # profitability
        'return_on_equity',
        'return_on_asset',
        "profit_before_tax",
        "net_profit",
        "net_profit_parent_company",
        "ebit",

        # leverage/liquidity
        'debt_to_equity',
        'current_ratio',
        
        # growth
        'revenue_growth_rate',
        'net_profit_growth_rate',

        # free cash flow
        'free_cash_flow_company_per_share',
        'free_cash_flow_company_per_share_ttm',

        # working capital
        "working_capital",
        "working_capital_lf",
        "working_capital_ttm",

        # investment
        "net_current_investment",

        # assets and liabilities
        "current_assets",
        "current_liabilities",

        "depreciation_and_amortization",

        "ebitda",

        # Balance
        "cash_equivalent", 
        "bill_receivable", 
        "net_accts_receivable", 
        "inventory", 
        "long_term_equity_investment", 
        "net_long_term_equity_investment", 
        "net_fixed_assets", 
        "intangible_assets", 
        "short_term_loans", 
        "long_term_liabilities_due_one_year", 
        "long_term_loans", 
        "bond_payable", 
        "long_term_payable", 
        "total_assets", 
        "equity_parent_company", 
        "total_equity",
    ]
    
    # Load models
    models = load_all_models()
    
    # Predict for a specific stock
    stock_to_predict = '601318.XSHG'  # Ping An Insurance
    
    results = predict_for_stock(
        models=models,
        stock_code=stock_to_predict,
        factors=factors,
        start_date='2022-01-01',
        end_date='2026-05-01'
    )
    
    # Display predictions
    display_predictions(results)
    
    # Display DCF valuations
    display_dcf_results(results, discount_rate=0.20, growth_rate=0.15)
    
    # Optional: Access individual predictions programmatically
    print(f"\n\n{'='*60}")
    print("PROGRAMMATIC ACCESS EXAMPLE")
    print(f"{'='*60}")
    
    rf_cashflows = results['predictions']['ffcf_random_forest'][0]
    print(f"\nRandom Forest 5-year cash flows: {rf_cashflows}")
    print(f"Random Forest Year 1: {rf_cashflows[0]:,.2f}")

/Users/tylerpruitt/Desktop/量化投资交易/Quant Trading Desk/.venv/lib/python3.8/site-packages/rqdatac/client.py:224: UserWarning: rqdatac is already inited. Settings will be changed.
  warnings.warn("rqdatac is already inited. Settings will be changed.", stacklevel=0)


Loading models...
✓ All models loaded successfully!

Fetching financial data for 601318.XSHG...

Cash Flow Predictions for 601318.XSHG

FFCF_RANDOM_FOREST:
----------------------------------------
  Year+1: 6.32
  Year+2: 8.88
  Year+3: 10.63
  Year+4: 11.99
  Year+5: 12.26

FFCF_GRADIENT_BOOSTING:
----------------------------------------
  Year+1: 26.24
  Year+2: 32.19
  Year+3: 35.02
  Year+4: 39.05
  Year+5: 34.51

FFCF_LSTM:
----------------------------------------
  Year+1: 5.96
  Year+2: -17.61
  Year+3: 3.24
  Year+4: 8.83
  Year+5: 6.84

DCF VALUATION (Discount Rate: 20.0%, Growth Rate: 15.0%)

FFCF_RANDOM_FOREST:
  DCF Value: 141.63

FFCF_GRADIENT_BOOSTING:
  DCF Value: 416.20

FFCF_LSTM:
  DCF Value: 64.88


PROGRAMMATIC ACCESS EXAMPLE

Random Forest 5-year cash flows: [ 6.32059532  8.88113733 10.63324343 11.99185119 12.26155208]
Random Forest Year 1: 6.32
